# Dealing with Imbalanced Data

*Imbalanced classes are a common problem in machine learning classification where there are a disproportionate ratio of observations in each class. Class imbalance can be found in many different areas including medical diagnosis, spam filtering, and fraud detection.*

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/creditcardfraud/creditcard.csv


In [2]:
data = pd.read_csv("/kaggle/input/creditcardfraud/creditcard.csv")
def display_all(data):
    with pd.option_context("display.max_row", 1000, "display.max_columns", 1000):
        display(data)
        
display_all(data.head())

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,-0.551600,-0.617801,-0.991390,-0.311169,1.468177,-0.470401,0.207971,0.025791,0.403993,0.251412,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,-0.166974,1.612727,1.065235,0.489095,-0.143772,0.635558,0.463917,-0.114805,-0.183361,-0.145783,-0.069083,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,0.207643,0.624501,0.066084,0.717293,-0.165946,2.345865,-2.890083,1.109969,-0.121359,-2.261857,0.524980,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,-0.054952,-0.226487,0.178228,0.507757,-0.287924,-0.631418,-1.059647,-0.684093,1.965775,-1.232622,-0.208038,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,0.753074,-0.822843,0.538196,1.345852,-1.119670,0.175121,-0.451449,-0.237033,-0.038195,0.803487,0.408542,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [3]:
data.Class.value_counts()

0    284315
1       492
Name: Class, dtype: int64

# The Problem with Imbalanced Classes

*Most machine learning algorithms work best when the number of samples in each class are about equal. This is because most algorithms are designed to maximize accuracy and reduce error.*

# The Problem with Accuracy
*Here we can use the DummyClassifier to always predict “not fraud” just to show how misleading accuracy can be.*

In [4]:
df1 = data.copy()

In [5]:
y = df1.Class
X = df1.drop("Class", axis = 1)

In [6]:
print(y.head())
print(X.head())

0    0
1    0
2    0
3    0
4    0
Name: Class, dtype: int64
   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V20       V21       V22       V23       V24  \
0  0.098698  0.363787  ...  0.251412 -0.018307  0.277838 -0.110474  0.066928   
1  0.085102 -0.255425  ... -0.069083 -0.225775 -0.638672  0.101288 -0.339846   
2  0.247676 -1.514654  ...  0.524980  0.247998  0.771679  0.909412 -0.689281   
3  0.377436 -1.387024  ... -0.208038 -0.108300  0.005274 -0.190321 -1.175575   
4 -0.270533  0.817739  ...  0.408542 -0.009431  0.79

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 27)
dummy_clf = DummyClassifier(strategy = "most_frequent")
dummy_clf.fit(X, y)
dummy_clf.predict(X)
dummy_clf.score(X, y)

0.9982725143693799

**We got an accuracy score of 99.8% — And without even training a model! Let’s compare this to logistic regression, an actual trained classifier.**

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

loj_model = LogisticRegression(solver = "liblinear").fit(X_train, y_train)
loj_pred = loj_model.predict(X_test)
accuracy_score(y_test, loj_pred)

0.9992135052386169

# 1. Change the performance metric
*As we saw above, accuracy is not the best metric to use when evaluating imbalanced datasets as it can be very misleading. Metrics that can provide better insight include:

* **Confusion Matrix:** a table showing correct predictions and types of incorrect predictions.
* **Precision:** the number of true positives divided by all positive predictions. Precision is also called Positive Predictive Value. It is a measure of a classifier’s exactness. Low precision indicates a high number of false positives.
* **Recall:** the number of true positives divided by the number of positive values in the test data. Recall is also called Sensitivity or the True Positive Rate. It is a measure of a classifier’s completeness. Low recall indicates a high number of false negatives.
* **F1: Score:** the weighted average of precision and recall.

**Let’s see what happens when we apply these F1 and recall scores to our logistic regression from above.**

In [9]:
from sklearn.metrics import f1_score,recall_score
print(f"f1 score: {f1_score(y_test, loj_pred)}")
print(f"recall_score: {recall_score(y_test, loj_pred)}")

f1 score: 0.7522123893805309
recall_score: 0.6439393939393939


**These scores don’t look quite so impressive. Let’s see what other methods we might try to improve our new metrics.**

In [10]:
df2 = df1.copy()

In [11]:
y = df2.Class
X = df2.drop("Class", axis = 1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=27)

# 2. Change the algorithm
*While in every machine learning problem, it’s a good rule of thumb to try a variety of algorithms, it can be especially beneficial with imbalanced datasets. Decision trees frequently perform well on imbalanced data. They work by learning a hierarchy of if/else questions and this can force both classes to be addressed.*

In [12]:
from sklearn.ensemble import RandomForestClassifier
rf_model = RandomForestClassifier(n_estimators = 10).fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
accuracy_score(y_test, rf_pred)

0.9996067526193084

In [13]:
print(f"f1 score: {f1_score(y_test, rf_pred)}")
print(f"recall_score: {recall_score(y_test,rf_pred)}")

f1 score: 0.8842975206611571
recall_score: 0.8106060606060606


**While our accuracy score is slightly lower, both F1 and recall have increased as compared to logistic regression! It appears that for this specific problem, random forest may be a better choice of model.**

# 3. Resampling Techniques — Oversample minority class
*Our next method begins our resampling techniques.
Oversampling can be defined as adding more copies of the minority class. Oversampling can be a good choice when you don’t have a ton of data to work with.
We will use the resampling module from Scikit-Learn to randomly replicate samples from the minority class.*

# Important Note
*Always split into test and train sets BEFORE trying oversampling techniques! Oversampling before splitting the data can allow the exact same observations to be present in both the test and train sets. This can allow our model to simply memorize specific data points and cause overfitting and poor generalization to the test data.*

In [14]:
from sklearn.utils import resample

y = data.Class
X = data.drop('Class', axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=27)


X = pd.concat([X_train, y_train], axis=1)


not_fraud = X[X.Class==0]
fraud = X[X.Class==1]


fraud_upsampled = resample(fraud,
                          replace=True,
                          n_samples=len(not_fraud),
                          random_state=27)

upsampled = pd.concat([not_fraud, fraud_upsampled])


upsampled.Class.value_counts()

1    213245
0    213245
Name: Class, dtype: int64

**After resampling we have an equal ratio of data points for each class! Let’s try our logistic regression again with the balanced training data.**

In [15]:
y_train = upsampled.Class
X_train = upsampled.drop('Class', axis=1)

upsampled = LogisticRegression(solver = "liblinear").fit(X_train, y_train)

upsampled_pred = upsampled.predict(X_test)
accuracy_score(y_test, upsampled_pred)

0.9807589674447347

In [16]:
print(f"f1 score: {f1_score(y_test, upsampled_pred)}")
print(f"recall_score: {recall_score(y_test,upsampled_pred)}")

f1 score: 0.14375000000000002
recall_score: 0.8712121212121212


**Our recall score increased, but F1 is much lower than with either our baseline logistic regression or random forest from above. Let’s see if undersampling might perform better here.**

# 4. Resampling techniques — Undersample majority class
*Undersampling can be defined as removing some observations of the majority class. Undersampling can be a good choice when you have a ton of data -think millions of rows. But a drawback is that we are removing information that may be valuable. This could lead to underfitting and poor generalization to the test set.
We will again use the resampling module from Scikit-Learn to randomly remove samples from the majority class.*

In [17]:
not_fraud_downsampled = resample(not_fraud,
                                replace = False, 
                                n_samples = len(fraud),
                                random_state = 27)

downsampled = pd.concat([not_fraud_downsampled, fraud])

downsampled.Class.value_counts()

1    360
0    360
Name: Class, dtype: int64

**Again, we have an equal ratio of fraud to not fraud data points, but in this case a much smaller quantity of data to train the model on. Let’s again apply our logistic regression.**

In [18]:
y_train = downsampled.Class
X_train = downsampled.drop('Class', axis=1)

undersampled = LogisticRegression(solver='liblinear').fit(X_train, y_train)

undersampled_pred = undersampled.predict(X_test)

accuracy_score(y_test, undersampled_pred)

0.9758574197354007

In [19]:
print(f"f1 score: {f1_score(y_test, undersampled_pred)}")
print(f"recall_score: {recall_score(y_test,undersampled_pred)}")

f1 score: 0.11710323574730355
recall_score: 0.8636363636363636


# 5. Generate synthetic samples
**A technique similar to upsampling is to create synthetic samples. Here we will use imblearn’s SMOTE or Synthetic Minority Oversampling Technique. SMOTE uses a nearest neighbors algorithm to generate new and synthetic data we can use for training our model.
Again, it’s important to generate the new samples only in the training set to ensure our model generalizes well to unseen data.**

In [21]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=27)
X_train, y_train = sm.fit_sample(X_train, y_train)

In [24]:
smote = LogisticRegression(solver = "liblinear").fit(X_train, y_train)
smote_pred = smote.predict(X_test)
accuracy_score(y_test, smote_pred)

0.9758574197354007

In [25]:
print(f"f1 score: {f1_score(y_test, smote_pred)}")
print(f"recall_score: {recall_score(y_test,smote_pred)}")

f1 score: 0.11710323574730355
recall_score: 0.8636363636363636


**Our F1 score is increased and recall is similar to the upsampled model above and for our data here outperforms undersampling.**